In [46]:
import pandas as pd
import datetime as dt 

In [47]:
sales_data = pd.read_csv('Downloads/Walmart.csv')
sales_data

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48
...,...,...,...,...,...,...,...,...,...,...,...
10046,9996,WALM056,Rowlett,Fashion accessories,$37,3.0,03/08/23,10:10:00,Cash,3.0,0.33
10047,9997,WALM030,Richardson,Home and lifestyle,$58,2.0,22/02/21,14:20:00,Cash,7.0,0.48
10048,9998,WALM050,Victoria,Fashion accessories,$52,3.0,15/06/23,16:00:00,Credit card,4.0,0.48
10049,9999,WALM032,Tyler,Home and lifestyle,$79,2.0,25/02/21,12:25:00,Cash,7.0,0.48


In [48]:
print("--- Initial Data Audit ---")
# Check initial data types and missing values
print(sales_data.info())
print("\nMissing values per column:")
print(sales_data.isnull().sum())

--- Initial Data Audit ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  object 
 2   City            10051 non-null  object 
 3   category        10051 non-null  object 
 4   unit_price      10020 non-null  object 
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  object 
 7   time            10051 non-null  object 
 8   payment_method  10051 non-null  object 
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 863.9+ KB
None

Missing values per column:
invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method

In [49]:
# This will show you all rows where quantity is missing
null_quantities = sales_data[sales_data['quantity'].isna()]
print(null_quantities)

      invoice_id   Branch           City             category unit_price  \
1892        1893  WALM024     Carrollton   Home and lifestyle        NaN   
1893        1894  WALM009          Plano  Fashion accessories        NaN   
1894        1895  WALM010         Laredo   Home and lifestyle        NaN   
1895        1896  WALM069       Rockwall  Fashion accessories        NaN   
1896        1897  WALM093       Angleton   Home and lifestyle        NaN   
1897        1898  WALM098  Mineral Wells  Fashion accessories        NaN   
1898        1899  WALM009          Plano   Home and lifestyle        NaN   
1899        1900  WALM071         Lufkin  Fashion accessories        NaN   
1900        1901  WALM090      Brownwood   Home and lifestyle        NaN   
1901        1902  WALM067    Haltom City  Fashion accessories        NaN   
1902        1903  WALM033       Pearland   Home and lifestyle        NaN   
1903        1904  WALM018         Frisco  Fashion accessories        NaN   
1904        

In [55]:
#Clean 'unit_price': Remove '$' and convert to float
if sales_data['unit_price'].dtype == 'object':
    sales_data['unit_price'] = sales_data['unit_price'].str.replace(r'[^\d.]', '', regex=True).astype(float)

In [56]:
sales_data['unit_price']

0        74.69
1        15.28
2        46.33
3        58.22
4        86.31
         ...  
10046    37.00
10047    58.00
10048    52.00
10049    79.00
10050    62.00
Name: unit_price, Length: 10020, dtype: float64

In [57]:
# Drop rows where values is missing
sales_data = sales_data.dropna(subset=['unit_price', 'quantity'])
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10020 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_id      10020 non-null  int64         
 1   Branch          10020 non-null  object        
 2   City            10020 non-null  object        
 3   category        10020 non-null  object        
 4   unit_price      10020 non-null  float64       
 5   quantity        10020 non-null  int64         
 6   date            10020 non-null  datetime64[ns]
 7   time            10020 non-null  object        
 8   payment_method  10020 non-null  object        
 9   rating          10020 non-null  float64       
 10  profit_margin   10020 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(5)
memory usage: 939.4+ KB


In [58]:
# Convert 'quantity' to integer
sales_data['quantity'] = sales_data['quantity'].astype(int)
sales_data['quantity']

0        7
1        5
2        7
3        8
4        7
        ..
10046    3
10047    2
10048    3
10049    2
10050    3
Name: quantity, Length: 10020, dtype: int64

In [59]:
# Convert 'date' column to a proper datetime object
sales_data['date'] = pd.to_datetime(sales_data['date'], format='%d/%m/%y', errors='coerce')
sales_data['date']

0       2019-01-05
1       2019-03-08
2       2019-03-03
3       2019-01-27
4       2019-02-08
           ...    
10046   2023-08-03
10047   2021-02-22
10048   2023-06-15
10049   2021-02-25
10050   2020-09-26
Name: date, Length: 10020, dtype: datetime64[ns]

In [63]:
# Creating the Total_Cost column
# Formula: Total_Cost = (UnitPrice * Quantity) * (1 - ProfitMargin)
sales_data['total_cost'] = (sales_data['unit_price'] * sales_data['quantity']) * (1 - sales_data['profit_margin'])

# Quick check to verify the new column
print("--- Sample of created Features and new column ---")
print(sales_data[['unit_price', 'quantity', 'profit_margin', 'total_cost']].head())

--- Sample of Engineered Features ---
   unit_price  quantity  profit_margin  total_cost
0       74.69         7           0.48    271.8716
1       15.28         5           0.48     39.7280
2       46.33         7           0.33    217.2877
3       58.22         8           0.33    312.0592
4       86.31         7           0.48    314.1684


In [70]:
# Calculate Gross Revenue per transaction for accurate transaction value profiling
sales_data['Transaction_Value'] = sales_data['unit_price'] * sales_data['quantity']

# Group by Payment Method
behavior_profile = sales_data.groupby('payment_method').agg(
    Average_Transaction_Value=('Transaction_Value', 'mean'),
    Average_Rating=('rating', 'mean'),
    Transaction_Count=('payment_method', 'count')
).reset_index()

print("--- Behavioral Profiling by payment Method ---")
print(behavior_profile.to_string(index=False))

--- Behavioral Profiling by payment Method ---
payment_method  Average_Transaction_Value  Average_Rating  Transaction_Count
          Cash                 142.767707        5.403191               1880
   Credit card                 114.840578        5.415731               4259
       Ewallet                 117.834597        6.476707               3881


In [71]:
sales_data.to_csv('new_walmart.csv')